# Lab 3.6: CI/CD Integration

**What you'll build:** A CI pipeline configuration using Claude Code's headless mode (`-p` flag) with structured JSON output -- and learn why session isolation and explicit context passing are essential for reproducible CI results.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- naive CI integration with inconsistent results | 5 min |
| 2 | The Right Way -- structured output, session isolation, multi-step pipelines | 5 min |
| 3 | Your Turn -- build a CI review pipeline with finding aggregation | 10 min |
| 4 | Stress Test -- validate pipeline produces consistent results | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are adding Claude Code to a GitHub Actions CI pipeline. The pipeline should:
1. Review each changed file in a PR for security issues
2. Output structured JSON findings
3. Post findings as PR comments
4. Optionally, generate fix suggestions

We'll simulate the CI pipeline using the Anthropic API (since we cannot run `claude` CLI directly in a notebook), but the patterns and architecture are identical to what you'd use with `claude -p`.

In [ ]:
# Simulated PR diff with security issues
PR_FILES = {
    "src/api/users.py": {
        "diff": '''
+def get_user(user_id):
+    query = f"SELECT * FROM users WHERE id = {user_id}"  # SQL injection!
+    result = db.execute(query)
+    return result.fetchone()
+
+def update_email(user_id, new_email):
+    # No input validation on email
+    db.execute("UPDATE users SET email = %s WHERE id = %s", (new_email, user_id))
+    return {"status": "updated", "email": new_email}
''',
        "known_issues": ["SQL injection in get_user via f-string", "No email validation in update_email"]
    },
    "src/api/auth.py": {
        "diff": '''
+def login(username, password):
+    user = db.query(User).filter_by(username=username).first()
+    if user and user.password == password:  # Plain text comparison!
+        token = jwt.encode({"user_id": user.id}, SECRET_KEY)
+        return {"token": token}
+    return {"error": f"Login failed for user {username}"}  # Info leakage
''',
        "known_issues": ["Plain text password comparison", "Username leakage in error message"]
    },
    "src/utils/logging.py": {
        "diff": '''
+import logging
+
+def log_request(request):
+    logging.info(f"Request: {request.method} {request.path}")
+    logging.debug(f"Headers: {request.headers}")  # May log auth tokens
+    logging.debug(f"Body: {request.body}")  # May log passwords
''',
        "known_issues": ["Logging headers may expose auth tokens", "Logging body may expose passwords"]
    }
}

print(f"=== SIMULATED PR ===")
print(f"Files changed: {len(PR_FILES)}")
for fname, fdata in PR_FILES.items():
    print(f"  {fname}: {len(fdata['known_issues'])} known issues")

---

## Phase 1: The Wrong Approach

A naive CI integration: review all files in one prompt, no structured output, no isolation.

In [ ]:
# Wrong: Single monolithic review of entire PR
all_diffs = "\n".join(f"--- {fname} ---\n{fdata['diff']}" 
                       for fname, fdata in PR_FILES.items())

# This simulates: claude -p "Review this PR..."
# Problems: no structure, attention dilution, unstructured output
NAIVE_PROMPT = f"""Review this PR for security issues. Be thorough.

{all_diffs}

List all issues you find."""

response = client.messages.create(
    model=MODEL,
    max_tokens=2000,
    messages=[{"role": "user", "content": NAIVE_PROMPT}]
)

naive_output = response.content[0].text.strip()
print("=== NAIVE CI OUTPUT ===")
print(naive_output[:800])
print("\n...")
print("\nPROBLEMS:")
print("  1. Unstructured text -- cannot be parsed by downstream scripts")
print("  2. All files reviewed at once -- attention dilution on large PRs")
print("  3. No JSON output -- cannot post as structured PR comments")
print("  4. 'Be thorough' is vague -- inconsistent results across runs")

---

## Phase 2: The Right Approach

Proper CI integration: per-file review, structured JSON output, session isolation.

In [ ]:
# Right: Per-file review with structured JSON output
# This simulates: claude -p "..." --output-format json
# Each file gets its own isolated review (separate API call = separate session)

REVIEW_PROMPT_TEMPLATE = """You are a security reviewer in a CI pipeline. Review this file diff
for security vulnerabilities.

File: {filename}
Diff:
```
{diff}
```

REVIEW CRITERIA -- flag ONLY when:
1. User input is used in SQL queries without parameterization (SQL injection)
2. Passwords are compared or stored in plain text
3. Sensitive data (passwords, tokens, API keys) appears in logs or error messages
4. Authentication/authorization checks are missing or bypassable

DO NOT flag:
- Missing input validation that is not a security risk
- Code style issues
- Performance concerns

Return a JSON array of findings. Each finding must have:
- "file": filename
- "line_hint": description of where in the file
- "severity": "critical", "high", "medium", or "low"
- "category": "sql_injection", "auth", "data_exposure", or "other"
- "description": what the issue is
- "recommendation": how to fix it

If no security issues are found, return an empty array [].
Return ONLY the JSON array."""

all_findings = []

for filename, filedata in PR_FILES.items():
    prompt = REVIEW_PROMPT_TEMPLATE.format(
        filename=filename,
        diff=filedata['diff']
    )
    
    # Each file gets its own API call (= session isolation)
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    try:
        findings = json.loads(raw)
        all_findings.extend(findings)
        print(f"  {filename}: {len(findings)} findings")
    except json.JSONDecodeError:
        print(f"  {filename}: Failed to parse JSON")

print(f"\n=== STRUCTURED CI OUTPUT ===")
print(f"Total findings: {len(all_findings)}")
for f in all_findings:
    print(f"  [{f.get('severity', '?').upper()}] {f.get('file', '?')}: {f.get('description', '?')[:80]}")

In [ ]:
# Aggregate findings into a CI report
ci_report = {
    "total_findings": len(all_findings),
    "by_severity": {},
    "by_category": {},
    "by_file": {},
    "findings": all_findings
}

for f in all_findings:
    sev = f.get('severity', 'unknown')
    cat = f.get('category', 'unknown')
    file = f.get('file', 'unknown')
    
    ci_report['by_severity'][sev] = ci_report['by_severity'].get(sev, 0) + 1
    ci_report['by_category'][cat] = ci_report['by_category'].get(cat, 0) + 1
    ci_report['by_file'][file] = ci_report['by_file'].get(file, 0) + 1

print("=== CI REPORT (JSON for downstream processing) ===")
print(json.dumps(ci_report, indent=2))

# This is what --output-format json would produce
print("\n--- In real CI, this JSON is piped to: ---")
print("  - PR comment posting script")
print("  - CI gate (fail if critical/high findings)")
print("  - Dashboard/metrics collection")

---

## Phase 3: Your Turn

Build a complete CI review pipeline that:
1. Reviews each file individually (session isolation)
2. Produces structured JSON findings
3. Generates fix suggestions for critical/high findings
4. Aggregates into a CI report with pass/fail decision

In [ ]:
# =============================================================
# TODO: Build your CI review pipeline
# =============================================================

def review_file(filename: str, diff: str) -> list:
    """
    Review a single file diff for security issues.
    Returns a list of structured findings.
    
    TODO: Write a review prompt with explicit criteria.
    Hint: Use the REVIEW_PROMPT_TEMPLATE from Phase 2 as a starting point.
    """
    # TODO: Implement this function
    raise NotImplementedError("Complete review_file()")


def generate_fixes(findings: list) -> list:
    """
    Generate fix suggestions for critical and high severity findings.
    This simulates a second CI step using --continue or prior findings injection.
    
    TODO: For each critical/high finding, ask Claude for a specific code fix.
    """
    # TODO: Implement this function
    raise NotImplementedError("Complete generate_fixes()")


def build_ci_report(findings: list, fixes: list, threshold: str = "high") -> dict:
    """
    Build a CI report with pass/fail decision.
    
    Args:
        findings: list of finding dicts
        fixes: list of fix suggestion dicts
        threshold: minimum severity to fail the build ("critical" or "high")
    
    Returns:
        CI report dict with pass/fail, findings, and fixes.
    
    TODO: Implement the report builder with pass/fail logic.
    """
    # TODO: Implement this function
    raise NotImplementedError("Complete build_ci_report()")


# Run the pipeline
try:
    pipeline_findings = []
    for fname, fdata in PR_FILES.items():
        file_findings = review_file(fname, fdata['diff'])
        pipeline_findings.extend(file_findings)
    
    pipeline_fixes = generate_fixes(pipeline_findings)
    report = build_ci_report(pipeline_findings, pipeline_fixes)
    
    print("=== YOUR CI PIPELINE REPORT ===")
    print(json.dumps(report, indent=2))
except NotImplementedError as e:
    print(f"TODO: {e}")
    print("Complete the three functions above and rerun.")

In [ ]:
# =============================================================
# Checker: validates your CI pipeline
# =============================================================

def check_pipeline(review_fn, fixes_fn, report_fn, pr_files):
    errors = []
    
    # Check 1: review_file returns structured findings
    try:
        test_findings = review_fn("test.py", "+ x = input()")
        if not isinstance(test_findings, list):
            errors.append("review_file must return a list")
        elif test_findings:  # If findings found
            for f in test_findings:
                required_keys = ['file', 'severity', 'description']
                for key in required_keys:
                    if key not in f:
                        errors.append(f"Finding missing '{key}' field")
                        break
    except NotImplementedError:
        errors.append("review_file not implemented")
    except Exception as e:
        errors.append(f"review_file raised: {e}")
    
    # Check 2: generate_fixes handles findings
    try:
        test_fix_input = [{"file": "test.py", "severity": "critical", 
                          "description": "SQL injection", "recommendation": "Use parameterized queries"}]
        fixes = fixes_fn(test_fix_input)
        if not isinstance(fixes, list):
            errors.append("generate_fixes must return a list")
    except NotImplementedError:
        errors.append("generate_fixes not implemented")
    except Exception as e:
        errors.append(f"generate_fixes raised: {e}")
    
    # Check 3: build_ci_report produces pass/fail
    try:
        test_report = report_fn(
            [{"severity": "critical", "file": "test.py", "description": "test"}],
            [],
            "high"
        )
        if not isinstance(test_report, dict):
            errors.append("build_ci_report must return a dict")
        elif 'pass' not in test_report and 'status' not in test_report and 'result' not in test_report:
            errors.append("CI report must contain a pass/fail decision")
    except NotImplementedError:
        errors.append("build_ci_report not implemented")
    except Exception as e:
        errors.append(f"build_ci_report raised: {e}")
    
    print("=== CI PIPELINE VALIDATION ===")
    if not errors:
        print("PASSED -- Pipeline produces structured, actionable output.")
    else:
        for e in errors:
            print(f"  [ERROR] {e}")
        print(f"\nFix {len(errors)} error(s) and try again.")
    
    return len(errors) == 0

check_pipeline(review_file, generate_fixes, build_ci_report, PR_FILES)

---

## Phase 4: Stress Test

Run the pipeline multiple times to verify consistency.

In [ ]:
# Run pipeline 3 times and compare results
print("Running pipeline 3 times for consistency...\n")

run_results = []
for run in range(3):
    try:
        findings = []
        for fname, fdata in PR_FILES.items():
            file_findings = review_file(fname, fdata['diff'])
            findings.extend(file_findings)
        
        # Track key metrics
        severities = [f.get('severity', '?') for f in findings]
        files_with_issues = set(f.get('file', '?') for f in findings)
        
        run_results.append({
            "total": len(findings),
            "critical": severities.count('critical'),
            "high": severities.count('high'),
            "files": len(files_with_issues)
        })
        print(f"  Run {run + 1}: {len(findings)} findings "
              f"(critical={severities.count('critical')}, high={severities.count('high')})")
    except NotImplementedError:
        print(f"  Run {run + 1}: SKIPPED (not implemented)")
        break
    except Exception as e:
        print(f"  Run {run + 1}: ERROR - {e}")

if len(run_results) == 3:
    totals = [r['total'] for r in run_results]
    consistent = max(totals) - min(totals) <= 1  # Allow 1 finding variance
    
    print(f"\n=== CONSISTENCY CHECK ===")
    print(f"Finding counts: {totals}")
    if consistent:
        print("Consistent -- pipeline produces stable results across runs.")
    else:
        print("Inconsistent -- tighten your review criteria for more deterministic output.")

### Key Takeaways

1. **`-p` flag enables headless mode** for CI. Each invocation gets session isolation by default.
2. **`--output-format json`** produces machine-readable output for downstream processing.
3. **Per-file review** prevents attention dilution on large PRs. Each file gets focused analysis.
4. **Session isolation is a feature.** Clean context = reproducible results. Use `--continue` only when you explicitly need prior context.
5. **Multi-step pipelines** can pass findings forward via JSON (explicit) or `--continue` (session-based).
6. **Explicit review criteria** in the prompt produce more consistent CI results than vague instructions.